# Assignment 3: Neural Networks for Quantum State Tomography

In this assignment you will train a neural network to fit the distribution of a quantum state from few measurements of it. Even though this problem has applications in physics, for the purpose of this exercise we will mainly consider it as a regression problem, without concerning us with the physics behind it.

> ⚠️ As usual, we will execute your notebook again when grade you. Please, make sure that it takes no longer than 15min (on google colab) to run from top to bottom. You should be able to solve the exercises without GPU acceleration, but feel free to add it if you want. Comment out code used for exploration where we ask you.


## Introducing the problem: Quantum State Tomography

We are given a dataset with $N$ samples $\{x_i,y_i \}_{i = 1, ..., N}$ where $x_i = \{0,1\}^{\mathtt{n\_spins}}$ is a binary string of length $\mathtt{n\_spins}$ and $y_i \in (0,1]$ is a  number. We have our regression problem.

But where is the data from? We are considering the case where the $x_i$ are the the possible outcomes of a measurement of a quantum state $\Psi$ and $y_i$ is the probability of observing outcome $x_i$.
Our goal is to train a neural network $f_\theta(x) = \hat{y}$ that approximates the original probability distribution. Specifically, for the set of possible outcomes $\{x_i\}$, the model should return a vector of probabilities corresponding to each outcome.


## Exercise 1 (10 pts): A fully connected network

For the purpose of this exercise, we give you a function that samples $N$ measurements from a quantum state along with their probabilities.

In [ ]:
import numpy as np
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

def get_data(n_spins):
    
    # Loads the data available in the .txt files we made available to you.
    
    if n_spins not in [8,12,16]:
        raise ValueError(f"The data for {n_spins} is not available.")
        
    numbers = [bin(i)[2:].zfill(n_spins) for i in range(2**n_spins)]
    xs = [[int(char) for char in string] for string in numbers]
    X = np.array(xs)
    
    Wstate = np.loadtxt(f"target_state_{n_spins}.txt")
    Y = np.abs(Wstate)**2
    
    # shuffle
    idx = np.random.permutation(range(len(Y)))
    
    return X[idx], Y[idx]

a) Load the data for `n_spins=16` and visualize the labels as a histogram. Visualize both `y` and `log(y)` in different plots. Which representation is more informative?

*Answer:* ...

In [ ]:
X, Y = ...

# your code here

b) We want to continue working with this datasets in two different forms, either without or with the log-transform of the labels : $\{x_i,y_i\}$ or $\{x_i,log(y_i)\}$. Then we want to train neural networks on this data using SGD. 

Complete the function `get_loaders`. Set `n_spins=12` and apply the log transform according to the normalize variable. Create a test, train and validation dataset using `TensorDataset`, the split should be done at (40%,20%,40%) of the original data. To prepare for training, wrap the new datasets in torch `DataLoaders` as you have seen in previous exercises. For training data loader, use a batch size 32 and use shuffling, for test and validation data loaders use the full batch and don't shuffle. 

In [ ]:
def get_loaders(X, Y, normalize_log = False):
    
    n_spins = 12
   
    ... # your code here
    
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = get_loaders(X, Y, normalize_log=True)

c) Define a general function `train` as given below that takes the test and train loaders, a learning rate and a model, a number of epochs and optionally a boolean `logging`equal to `True` by default and returns the train and test losses for every epoch, and the best test loss achieved overall. 

You are supposed to run SGD with the MSE loss. Print the progress of your training by printing the losses every epoch, if `logging==True`.

- I) In the course of your code you will use the line `optimizer.zero_grad()`. Explain briefly what happens when you call this function, and when and why you need to call it.

- II)  Explain why we previously needed to activate the batches and shuffling for the training dataloader. 

- III) When a dataloader is used several times for several epochs, is the data shuffled in the same order every time?

*`Answers:`* ...

In [ ]:
def train(train_loader, test_loader, lr, model, n_epochs, logging=True):
    
    # your code here
    ...
    
    return train_losses, test_losses, best_test_loss

d) Test your function by training a 1 hidden layer fully connected network with ReLU activations. You will choose the hyperparameters: `n_epochs` to train and the `learning rate`. You can keep the number of `hidden_neurons=16` fixed for the moment. Hint: You should not need to use more than 2000 epochs in your explorations...

Write the code to train and visualize the test and train losses with a log-sacle on the y-axis.
It is now your turn to play.

Choose `lr=0.1`, `n_epochs=100` and `hidden_neurons=16`. Train 10 models with these hyperparameters and observe their learning curves. We would expect the learning curves of the 10 different models to be different because every time you define a new model, the weights are initialized randomly and the SGD optimizer will go over the batches in a different order. 

Describe what differences and similarities you observe. How could you change (some of) the hyperparameters so that the runs are more similar, i.e. the loss curve is less strongly dependent on the initialization?

> ⚠️ Once you are ready to answer the question, leave the code you used for your explorations and the learning curves you produced, but comment it so we don't have to run it again. This hold only for this question

*`Answer:`* 

In [ ]:
"""
# your code here after you ran it to answer the question, in a comment
"""

e) Choose some parameters that reliably give you good models even if you restart the training . If you get a test loss around 0.01 you can stop training. 

*`Answer:`* ..

In [ ]:
# your code here

f) For the previous model that you liked, plot the histogram of the distribution again with the log transform, for test data. Also plot the predicted and ground truth values against each other. Are you satisfied with the match?

*`Answer:`* ...

In [ ]:
# your code here

## Exercise 2 (6 pts): Exploring alternative architectures  and losses

You have found a model that works! But can you make it more efficient?

a) A linear regression model would for example use fewer parameters. Try the model and based on your experiment argue that it is less suitable for this task than the fully connected neural network.

*`Answer:`* ...

In [ ]:
# your code here

b) If a linear regression does not work, maybe you can fine-tune the number of hidden neurons. Select 5 different values of the hidden neurons `[2,8,16,32,64]` and train those models. Use at max 500 epochs to train every model and a learning rate of 0.01. In reality you would optimize the learning rate for each architecture and you can still do so if you wish, but to save you time we do not require this. After the models are trained, select the best one based on the best test loss. 

Estimate the loss you can expect on the model that you just selected on a fresh data sample, that has not been seen in the process of selecting this model. 

*`Answer:`* ...

In [ ]:
# your code here

c) Now you want to see what difference the log transformation we applied at the very beginning makes. Train a model on the plain data and compare the models prediction again both via the histogram of y-values and the scatterplot between the predicted and true y, in the log transform. What did the non-log transformed model learn and why?

*`Answer:`* ...

In [ ]:
# your code here